In [37]:
import pyxdf
import pandas as pd

In [50]:
data, header = pyxdf.load_xdf("sub-002_ses-002_task-Default_run-001_eeg.xdf")
print("Streams found:", [s['info']['name'][0] for s in new_order])
print("First stream sample shape:", new_order[0]['time_series'])

found likely XDF file corruption (unpack requires a buffer of 8 bytes), scanning forward to next boundary chunk.


Streams found: ['EEG_gpype', 'PsychoPyMarkers', 'EEG_LIFU_events', 'PsychoPy_numeric']
First stream sample shape: [[-4.88332734e+04  1.33997440e+00  1.79553127e+00 ... -1.82936922e+05
  -1.61064047e+05 -4.56808867e+04]
 [-4.12112500e+04  1.48122919e+00  2.19403982e+00 ... -1.80806672e+05
  -1.59896969e+05 -4.52630820e+04]
 [-3.59786484e+04  1.59303701e+00  2.53776693e+00 ... -1.79868719e+05
  -1.59786828e+05 -4.58892383e+04]
 ...
 [ 2.55839629e+04 -5.66350460e-01  3.20752800e-01 ... -4.63649297e+04
   1.22200986e+04  1.12301609e+05]
 [ 0.00000000e+00 -1.59332916e-01  2.53869798e-02 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 1.27428545e+04  2.32060552e-01  5.38520999e-02 ... -5.54153438e+04
   4.57434668e+03  1.00452375e+05]]


In [49]:
import pyxdf

# Load the original XDF
streams, file_header = pyxdf.load_xdf("sub-test_ses-test_task-Default_run-001_eeg.xdf")

# Helper: get stream by name
def get_stream(name):
    for s in streams:
        if s['info']['name'][0] == name:
            return s
    raise ValueError(f"Stream '{name}' not found")

# Rearrange order (EEG first)
new_order = [
    get_stream("EEG_gpype"),               # your EEG stream name
    get_stream("PsychoPyMarkers"),   # example
    get_stream("EEG_LIFU_events"),   # example
    get_stream("PsychoPy_numeric"),  # example
]

new_order


[{'info': defaultdict(list,
              {'name': ['EEG_gpype'],
               'type': ['EEG'],
               'channel_count': ['13'],
               'channel_format': ['double64'],
               'source_id': ['EEG_gpype'],
               'nominal_srate': ['250.0000000000000'],
               'version': ['1.100000000000000'],
               'created_at': ['2048074.908516800'],
               'uid': ['39367794-98ae-4482-b33b-81702b164f37'],
               'session_id': ['default'],
               'hostname': ['JanetsPC'],
               'v4address': [None],
               'v4data_port': ['16574'],
               'v4service_port': ['16592'],
               'v6address': [None],
               'v6data_port': ['16574'],
               'v6service_port': ['16592'],
               'desc': [None],
               'stream_id': 3,
               'effective_srate': 249.97733840976946,
               'segments': [(0, 15552)],
               'clock_segments': [(0, 15552)]}),
  'footer': {'info': 

In [38]:
import matplotlib.pyplot as plt
data[3]['time_series'].shape
df = pd.DataFrame(data[3]['time_series'])
df['timestamp'] = data[3]['time_stamps']
df['timestamp'].diff() 


0          NaN
1        0.004
2        0.004
3        0.004
4        0.004
         ...  
15548    0.004
15549    0.004
15550    0.004
15551    0.004
15552    0.004
Name: timestamp, Length: 15553, dtype: float64

In [51]:
import pandas as pd


dfs = []
for stream in new_order:
    name = stream['info']['name'][0]
    df = pd.DataFrame(stream['time_series'])
    df['timestamp'] = stream['time_stamps']
    # Drop rows with missing timestamps
    df = df.dropna(subset=['timestamp'])
    # Add stream name prefix to columns (except timestamp)
    df = df.rename(columns=lambda c: f"{name}_{c}" if c != 'timestamp' else c)
    dfs.append(df.sort_values('timestamp'))

# Start with the first stream
merged = dfs[0]

# Merge the rest one by one on 'timestamp'
for df in dfs[1:]:
    merged = pd.merge_asof(
        merged,
        df,
        on='timestamp',
        direction="nearest",
        tolerance=0.001  # adjust tolerance in seconds
    )

# Show first 20 rows
merged


,EEG_gpype_0,EEG_gpype_1,EEG_gpype_2,EEG_gpype_3,EEG_gpype_4,EEG_gpype_5,EEG_gpype_6,EEG_gpype_7,EEG_gpype_8,EEG_gpype_9,EEG_gpype_10,EEG_gpype_11,EEG_gpype_12,timestamp,PsychoPyMarkers_0,EEG_LIFU_events_0,PsychoPy_numeric_0
0,-48833.273438,1.339974,1.795531,1.586273,-0.175533,-0.187872,-0.187959,-200000.000000,-200000.000000,-200000.000000,-182936.921875,-161064.046875,-45680.886719,2.048174e+06,NaN,NaN,NaN
1,-41211.250000,1.481229,2.194040,1.629554,-0.165178,-0.187872,-0.224415,-200000.000000,-200000.000000,-200000.000000,-180806.671875,-159896.968750,-45263.082031,2.048174e+06,NaN,NaN,NaN
2,-35978.648438,1.593037,2.537767,1.680248,-0.153051,-0.187872,-0.225102,-200000.000000,-200000.000000,-200000.000000,-179868.718750,-159786.828125,-45889.238281,2.048174e+06,NaN,NaN,NaN
3,-43072.367188,1.675236,2.806414,1.734780,-0.140005,-0.187872,-0.162589,-200000.000000,-200000.000000,-200000.000000,-181862.250000,-160903.921875,-46314.269531,2.048174e+06,NaN,NaN,NaN
4,-49462.113281,1.728805,2.988765,1.789520,-0.126909,-0.187872,-0.077550,-200000.000000,-200000.000000,-200000.000000,-183153.171875,-161208.843750,-45776.156250,2.048174e+06,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15548,17467.833984,-1.356120,1.839062,3.183557,0.206593,0.197444,0.747910,-86045.625000,-45688.113281,-87133.937500,-48837.332031,6879.831055,109499.960938,2.048236e+06,NaN,NaN,NaN
15549,19867.041016,-0.968364,0.937728,3.176766,0.204968,0.197444,0.637720,-85092.539062,-44597.054688,-84976.398438,-47854.550781,8516.359375,110534.039062,2.048236e+06,NaN,NaN,NaN
15550,25583.962891,-0.566350,0.320753,3.160745,0.201135,0.197444,0.424945,-83634.648438,-46225.023438,-81441.687500,-46364.929688,12220.098633,112301.609375,2.048236e+06,NaN,NaN,NaN
15551,0.000000,-0.159333,0.025387,3.142478,0.196765,0.197444,-1.601326,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.048236e+06,NaN,NaN,NaN


In [45]:
# Keep only rows where EEG_gpype_1 is not NaN
eeg_data = merged[merged['EEG_gpype_0'].notna()]
eeg_timestamps = eeg_data['timestamp']
eeg_data # Check the time differences between consecutive samples


,EEG_gpype_0,EEG_gpype_1,EEG_gpype_2,EEG_gpype_3,EEG_gpype_4,EEG_gpype_5,EEG_gpype_6,EEG_gpype_7,EEG_gpype_8,EEG_gpype_9,EEG_gpype_10,EEG_gpype_11,EEG_gpype_12,timestamp,PsychoPyMarkers_0,EEG_LIFU_events_0,PsychoPy_numeric_0
0,-48833.273438,1.339974,1.795531,1.586273,-0.175533,-0.187872,-0.187959,-200000.000000,-200000.000000,-200000.000000,-182936.921875,-161064.046875,-45680.886719,2.048174e+06,NaN,NaN,NaN
1,-41211.250000,1.481229,2.194040,1.629554,-0.165178,-0.187872,-0.224415,-200000.000000,-200000.000000,-200000.000000,-180806.671875,-159896.968750,-45263.082031,2.048174e+06,NaN,NaN,NaN
2,-35978.648438,1.593037,2.537767,1.680248,-0.153051,-0.187872,-0.225102,-200000.000000,-200000.000000,-200000.000000,-179868.718750,-159786.828125,-45889.238281,2.048174e+06,NaN,NaN,NaN
3,-43072.367188,1.675236,2.806414,1.734780,-0.140005,-0.187872,-0.162589,-200000.000000,-200000.000000,-200000.000000,-181862.250000,-160903.921875,-46314.269531,2.048174e+06,NaN,NaN,NaN
4,-49462.113281,1.728805,2.988765,1.789520,-0.126909,-0.187872,-0.077550,-200000.000000,-200000.000000,-200000.000000,-183153.171875,-161208.843750,-45776.156250,2.048174e+06,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15548,17467.833984,-1.356120,1.839062,3.183557,0.206593,0.197444,0.747910,-86045.625000,-45688.113281,-87133.937500,-48837.332031,6879.831055,109499.960938,2.048236e+06,NaN,NaN,NaN
15549,19867.041016,-0.968364,0.937728,3.176766,0.204968,0.197444,0.637720,-85092.539062,-44597.054688,-84976.398438,-47854.550781,8516.359375,110534.039062,2.048236e+06,NaN,NaN,NaN
15550,25583.962891,-0.566350,0.320753,3.160745,0.201135,0.197444,0.424945,-83634.648438,-46225.023438,-81441.687500,-46364.929688,12220.098633,112301.609375,2.048236e+06,NaN,NaN,NaN
15551,0.000000,-0.159333,0.025387,3.142478,0.196765,0.197444,-1.601326,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.048236e+06,NaN,NaN,NaN


In [52]:
eeg_timestamps.diff().head(50)

0       NaN
1     0.004
2     0.004
3     0.004
4     0.004
5     0.004
6     0.004
7     0.004
8     0.004
9     0.004
10    0.004
11    0.004
12    0.004
13    0.004
14    0.004
15    0.004
16    0.004
17    0.004
18    0.004
19    0.004
20    0.004
21    0.004
22    0.004
23    0.004
24    0.004
25    0.004
26    0.004
27    0.004
28    0.004
29    0.004
30    0.004
31    0.004
32    0.004
33    0.004
34    0.004
35    0.004
36    0.004
37    0.004
38    0.004
39    0.004
40    0.004
41    0.004
42    0.004
43    0.004
44    0.004
45    0.004
46    0.004
47    0.004
48    0.004
49    0.004
Name: timestamp, dtype: float64